In [1]:
# upgrading scikit-learn to V1.8.0 because django project has this version
import sys
!{sys.executable} -m pip install -i https://mirror-pypi.runflare.com/simple --upgrade scikit-learn==1.8.0

Looking in indexes: https://mirror-pypi.runflare.com/simple


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import joblib

In [3]:
BASE_DIR = Path.cwd().parent    # 'Trained Model' folder
ROOT = BASE_DIR.parent    # 'HumanAI Income Lab' folder

sys.path.append(str(ROOT/'income_lab'))
# Python can only import modules from locations listed in sys.path

from income_predictor.preprocessing import AdultPreprocessor

df = pd.read_csv(BASE_DIR/'adult.csv')
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


# Preprocessing

In [4]:
df['income'] = df['income'].str.lower().str.strip().map({'<=50k': 0,
                                                       '>50k': 1})

y = df['income']
X = df.drop(columns=['income'])

numeric_cols = [
    'age', 'hours_per_week', 'education_num', 'capital_net'
]

categorical_cols = [
    'workclass', 'marital_status', 'occupation',
    'relationship', 'race', 'sex'
    # native_country is handled separately through freq mapping
]

# Building the ColumnTransformer

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
    ],
    remainder='drop'
)

# Full Pipeline

In [6]:
pipeline = Pipeline(
    steps=[
        ('cleaning', AdultPreprocessor()),
        ('encode_scale', preprocessor),
        ('NN_model', MLPClassifier(
            hidden_layer_sizes= (64,),
            activation='relu',
            solver='adam',
            alpha=0.001,
            random_state=42
        ))
    ]
)

# Splitting

In [7]:
x_tr, x_ts, y_tr, y_ts = train_test_split(X, y,
                                         test_size=0.2)

# Training & Evaluation

In [8]:
pipeline.fit(x_tr, y_tr)

preds = pipeline.predict(x_ts)

print("Accuracy:", accuracy_score(y_ts, preds))
print("Precision:", precision_score(y_ts, preds))
print("Recall:", recall_score(y_ts, preds))
print("F1 Score:", f1_score(y_ts, preds))

C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Accuracy: 0.8542914171656687
Precision: 0.6963946869070209
Recall: 0.7012738853503184
F1 Score: 0.6988257695969533


# Export

In [9]:
joblib.dump(pipeline, BASE_DIR/"Pipeline nb"/"NN_model_pipeline.pkl")

print('File "NN_model_pipeline.pkl" exported.')

File "NN_model_pipeline.pkl" exported.
